# Sequences and Series of Functions — An Interactive Laboratory

**Companion notebook for Chapter 3 of _Advanced Calculus of One Variable with Python and AI_.**

Pointwise convergence fixes the point first, whereas uniform convergence controls the entire domain with one index:

$$
\forall x\,\forall\varepsilon\,\exists N(\varepsilon,x)
\qquad\text{versus}\qquad
\forall\varepsilon\,\exists N(\varepsilon)\,\forall x.
$$

This notebook turns the chapter's main theorems and counterexamples into controlled experiments. Numerical sampling is used to discover structure, but exact suprema, theorem hypotheses, and witness sequences remain the basis of every rigorous conclusion.

### Learning goals

1. Compare pointwise, uniform, and locally uniform convergence.
2. Use the supremum norm and witness sequences effectively.
3. Understand the uniform Cauchy criterion, Dini's theorem, and equicontinuity.
4. Test when limits may pass through integrals and derivatives.
5. Explore function series through the Weierstrass $M$-test and uniform Dirichlet test.
6. Study cumulative Riemann--Stieltjes integrals with varying integrators.
7. Recognize when a finite grid hides a moving or narrowing peak.

Run the cells from top to bottom. Every laboratory has a blue action button.

## Cell 1 — Imports, display style, and reusable helpers

Uniform convergence is measured by the largest error

$$
d_n=\sup_{x\in E}|f_n(x)-f(x)|.
$$

The setup cell loads the numerical, plotting, and widget libraries. If a package is missing, it is installed once in the active notebook environment. All later cells use the same visual style and accessible action buttons.

In [ ]:
import sys
import subprocess
import importlib


# Install only libraries that are missing from the current notebook runtime.
required_packages = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "ipywidgets": "ipywidgets",
    "IPython": "ipython",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import math
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown, HTML, clear_output
from scipy.integrate import cumulative_trapezoid


plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "blue": "#2563eb",
    "orange": "#ea580c",
    "green": "#15803d",
    "red": "#dc2626",
    "purple": "#7e22ce",
    "gray": "#475569",
}


def style_button(button, icon="play"):
    """Apply a consistent style to every action button."""
    button.button_style = "primary"
    button.icon = icon
    button.layout = widgets.Layout(width="185px")
    return button


def show_note(text, color="#eff6ff", border=None):
    """Display a short theorem, warning, or interpretation note."""
    border_color = border or COLORS["blue"]
    display(
        HTML(
            f"<div style='padding:10px 14px;border-left:4px solid {border_color};"
            f"background:{color};margin:8px 0'>{text}</div>"
        )
    )


display(
    HTML(
        "<div style='padding:12px 16px;background:#ecfdf5;border-left:5px solid #15803d'>"
        "<b>Setup complete.</b> The uniform-convergence laboratories are ready.</div>"
    )
)

## Cell 2 — Pointwise versus uniform convergence through the supremum error

The practical criterion is

$$
f_n\rightrightarrows f\text{ on }E
\quad\Longleftrightarrow\quad
\|f_n-f\|_{\infty,E}\longrightarrow0.
$$

The domain is part of the statement. For example, $x^n\to0$ uniformly on $[0,r]$ when $r<1$, but on $[0,1]$ the pointwise limit equals $0$ on $[0,1)$ and $1$ at the endpoint, and the convergence is not uniform. Compare this with uniform majorants such as $A^2/n$ and $1/n$.

In [ ]:
sup_family = widgets.Dropdown(
    options=[
        ("Powers: f_n(x)=x^n", "powers"),
        ("Rational approximation: nx/(n+x) -> x", "rational"),
        ("Polynomial perturbation: x+x^2/n -> x", "polynomial"),
        ("Shrinking oscillation: sin(nx+n)/n -> 0", "oscillation"),
    ],
    value="powers",
    description="Family:",
    layout=widgets.Layout(width="500px"),
)
sup_n = widgets.IntSlider(
    value=12, min=1, max=150, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
sup_domain = widgets.FloatSlider(
    value=0.90, min=0.10, max=1.00, step=0.01, description="r:",
    continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="500px")
)
sup_button = style_button(widgets.Button(description="Measure max error"), "ruler")
sup_output = widgets.Output()


def supremum_example(family, n, domain_parameter, grid_size=4001):
    """Return grid, approximation, limit, and the exact supremum error."""
    if family == "powers":
        r = domain_parameter
        x = np.linspace(0.0, r, grid_size)
        f_n = x**n
        if abs(r - 1.0) < 1e-12:
            f = np.zeros_like(x)
            f[-1] = 1.0
            exact_error = 1.0  # supremum on [0,1), not attained
        else:
            f = np.zeros_like(x)
            exact_error = r**n
        title = rf"$x^n$ on $[0,{r:.2f}]$"
    elif family == "rational":
        A = domain_parameter
        x = np.linspace(0.0, A, grid_size)
        f_n = n * x / (n + x)
        f = x
        exact_error = A**2 / (n + A)
        title = rf"$nx/(n+x)$ on $[0,{A:.2f}]$"
    elif family == "polynomial":
        A = domain_parameter
        x = np.linspace(0.0, A, grid_size)
        f_n = x + x**2 / n
        f = x
        exact_error = A**2 / n
        title = rf"$x+x^2/n$ on $[0,{A:.2f}]$"
    else:
        A = domain_parameter
        x = np.linspace(-A, A, grid_size)
        f_n = np.sin(n * x + n) / n
        f = np.zeros_like(x)
        exact_error = 1.0 / n  # exact on R; a valid uniform majorant here
        title = rf"$\sin(nx+n)/n$ on $[-{A:.2f},{A:.2f}]$"
    return x, f_n, f, exact_error, title


def run_supremum_lab(_=None):
    with sup_output:
        clear_output(wait=True)
        x, f_n, f, exact_error, title = supremum_example(
            sup_family.value, sup_n.value, sup_domain.value
        )
        grid_error = float(np.max(np.abs(f_n - f)))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(x, f_n, color=COLORS["blue"], lw=2, label=rf"$f_{{{sup_n.value}}}$")
        axes[0].plot(x, f, color=COLORS["red"], ls="--", lw=1.8, label="$f$")
        axes[0].set_title(title)
        axes[0].legend()
        axes[1].plot(x, np.abs(f_n - f), color=COLORS["orange"], lw=2)
        axes[1].axhline(exact_error, color=COLORS["red"], ls="--", label="exact supremum or majorant")
        axes[1].set_title(r"Error $|f_n-f|$")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Grid maximum error       = {grid_error:.10g}")
        print(f"Exact supremum/majorant  = {exact_error:.10g}")
        if sup_family.value == "powers" and abs(sup_domain.value - 1.0) < 1e-12:
            show_note(
                "The true supremum is 1 although no point in $[0,1)$ attains it. The pointwise limit is discontinuous, so uniform convergence is impossible.",
                "#fff7ed", COLORS["orange"]
            )
        else:
            show_note("An exact bound that tends to zero proves uniform convergence on the displayed fixed domain.")


def configure_supremum(change=None):
    family = sup_family.value
    if family == "powers":
        sup_domain.description = "r:"
        sup_domain.min, sup_domain.max, sup_domain.step = 0.10, 1.00, 0.01
        if sup_domain.value > 1.0:
            sup_domain.value = 0.90
    elif family == "oscillation":
        sup_domain.description = "A:"
        sup_domain.min, sup_domain.max, sup_domain.step = 1.0, 8.0, 0.25
        if sup_domain.value < 1.0:
            sup_domain.value = math.pi
    else:
        sup_domain.description = "A:"
        sup_domain.min, sup_domain.max, sup_domain.step = 0.25, 5.0, 0.25
        if sup_domain.value < 0.25:
            sup_domain.value = 1.0


sup_family.observe(configure_supremum, names="value")
sup_button.on_click(run_supremum_lab)
configure_supremum()
display(widgets.VBox([sup_family, sup_n, sup_domain, sup_button, sup_output]))
run_supremum_lab()

## Cell 3 — Witness sequences and the danger of a finite grid

To disprove uniform convergence, it is enough to find $x_n\in E$ and $\varepsilon_0>0$ such that

$$
|f_n(x_n)-f(x_n)|\ge\varepsilon_0.
$$

A finite grid may miss precisely these points. The triangular spike has support $[0,1/n]$, height $n$, and peak at $x_n=1/(2n)$. The smooth boundary spike has its maximum at $x_n=1/(n+1)$. For $x^n$ on $[0,1]$, the supremum error is $1$ even though the endpoint itself has zero error.

In [ ]:
spike_family = widgets.Dropdown(
    options=[
        ("Continuous triangular spike", "triangle"),
        ("Smooth boundary spike: n^2 x(1-x)^n", "boundary"),
        ("Endpoint layer: x^n on [0,1]", "powers"),
    ],
    value="triangle",
    description="Example:",
    layout=widgets.Layout(width="500px"),
)
spike_n = widgets.IntSlider(
    value=60, min=2, max=500, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
spike_grid_size = widgets.IntSlider(
    value=101, min=11, max=2001, step=10, description="Grid points:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
spike_button = style_button(widgets.Button(description="Test the grid"), "exclamation-triangle")
spike_output = widgets.Output()


def triangular_spike(x, n):
    """Continuous triangular spike with base 1/n and height n."""
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    left = (0.0 <= x) & (x <= 1.0 / (2.0 * n))
    right = (1.0 / (2.0 * n) < x) & (x <= 1.0 / n)
    y[left] = 2.0 * n**2 * x[left]
    y[right] = 2.0 * n - 2.0 * n**2 * x[right]
    return y


def spike_data(family, n, x):
    if family == "triangle":
        values = triangular_spike(x, n)
        witness = 1.0 / (2.0 * n)
        exact = float(n)
    elif family == "boundary":
        values = n**2 * x * (1.0 - x) ** n
        witness = 1.0 / (n + 1.0)
        exact = n**2 / (n + 1.0) * (n / (n + 1.0)) ** n
    else:
        # The pointwise limit is zero for x<1 and equals one at x=1.
        values = np.where(np.isclose(x, 1.0), 0.0, x**n)
        witness = 1.0 - 1.0 / n
        exact = 1.0
    return values, witness, exact


def run_spike_lab(_=None):
    with spike_output:
        clear_output(wait=True)
        n = spike_n.value
        family = spike_family.value
        coarse_x = np.linspace(0.0, 1.0, spike_grid_size.value)
        dense_x = np.linspace(0.0, 1.0, max(20001, 80 * n))
        coarse_y, witness, exact = spike_data(family, n, coarse_x)
        dense_y, _, _ = spike_data(family, n, dense_x)

        sampled = float(np.max(coarse_y))
        witness_value = float(spike_data(family, n, np.array([witness]))[0][0])

        fig, ax = plt.subplots(figsize=(10.5, 4.3))
        ax.plot(dense_x, dense_y, color=COLORS["blue"], lw=2, label="true error profile")
        ax.scatter(coarse_x, coarse_y, color=COLORS["orange"], s=18, label="sampled grid")
        ax.scatter([witness], [witness_value], color=COLORS["red"], s=70, zorder=4, label="$x_n$ witness")
        if family != "powers":
            ax.set_xlim(0, min(1.0, max(5.0 / n, 0.08)))
        ax.set_title("A finite grid can underestimate the supremum")
        ax.set_xlabel("$x$")
        ax.legend()
        plt.show()

        print(f"Exact supremum error      = {exact:.10g}")
        print(f"Maximum seen on the grid = {sampled:.10g}")
        print(f"Witness point x_n         = {witness:.10g}")
        print(f"Error at the witness      = {witness_value:.10g}")
        show_note("A sampled maximum is always a lower bound for the true supremum. It is never, by itself, a proof of uniform convergence.", "#fef2f2", COLORS["red"])


spike_button.on_click(run_spike_lab)
display(widgets.VBox([spike_family, spike_n, spike_grid_size, spike_button, spike_output]))

## Cell 4 — The uniform Cauchy criterion without knowing the limit

A sequence $(f_n)$ converges uniformly if and only if

$$
\forall\varepsilon>0\;\exists N\;\forall m,n\ge N:
\|f_m-f_n\|_\infty<\varepsilon.
$$

This is the Cauchy condition in the Banach space $\mathcal B(E)$. The experiment compares $f_n$ with $f_m$ for $m=qn$. One pair is not a proof, but persistent pairwise separation supplies evidence for a witness against the uniform Cauchy property.

In [ ]:
cauchy_family = widgets.Dropdown(
    options=[
        ("f_n(x)=x/n on [0,1]", "linear"),
        ("f_n(x)=x^n on [0,r]", "power_compact"),
        ("f_n(x)=x^n on [0,1]", "power_full"),
    ],
    value="power_full",
    description="Sequence:",
    layout=widgets.Layout(width="480px"),
)
cauchy_n = widgets.IntSlider(
    value=20, min=1, max=300, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
cauchy_factor = widgets.IntSlider(
    value=2, min=2, max=6, step=1, description="m/n:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
cauchy_r = widgets.FloatSlider(
    value=0.90, min=0.20, max=0.99, step=0.01, description="r:",
    continuous_update=False, layout=widgets.Layout(width="480px")
)
cauchy_button = style_button(widgets.Button(description="Compare late terms"), "exchange-alt")
cauchy_output = widgets.Output()


def power_pair_supremum(n, m, r):
    """Exact maximum of x^n-x^m on [0,r] for m>n."""
    critical = (n / m) ** (1.0 / (m - n))
    x_star = min(r, critical)
    return x_star**n - x_star**m, x_star


def run_uniform_cauchy(_=None):
    with cauchy_output:
        clear_output(wait=True)
        n = cauchy_n.value
        m = cauchy_factor.value * n
        family = cauchy_family.value
        r = 1.0 if family == "power_full" else cauchy_r.value
        x = np.linspace(0.0, r, 5001)

        if family == "linear":
            f_n = x / n
            f_m = x / m
            exact = abs(1.0 / n - 1.0 / m)
            witness = 1.0
        else:
            f_n = x**n
            f_m = x**m
            exact, witness = power_pair_supremum(n, m, r)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(x, f_n, color=COLORS["blue"], label=rf"$f_{{{n}}}$")
        axes[0].plot(x, f_m, color=COLORS["orange"], label=rf"$f_{{{m}}}$")
        axes[0].set_title("Two late functions")
        axes[0].legend()
        axes[1].plot(x, np.abs(f_n - f_m), color=COLORS["purple"], lw=2)
        axes[1].scatter([witness], [exact], color=COLORS["red"], s=60, zorder=3)
        axes[1].set_title(r"Pairwise distance $|f_n-f_m|$")
        plt.tight_layout()
        plt.show()

        print(f"m                            = {m}")
        print(f"Exact pairwise sup distance = {exact:.10g}")
        print(f"Maximizing/witness point    = {witness:.10g}")
        if family == "power_full" and cauchy_factor.value == 2:
            show_note("For $m=2n$, the exact distance is $1/4$ for every $n$. Hence the sequence is not uniformly Cauchy on $[0,1]$.", "#fef2f2", COLORS["red"])
        else:
            show_note(r"Observe the rate as $n$ grows. A proof must control every pair $m,n\ge N$, not only the selected ratio.")


def configure_cauchy(change=None):
    cauchy_r.disabled = cauchy_family.value != "power_compact"


cauchy_family.observe(configure_cauchy, names="value")
cauchy_button.on_click(run_uniform_cauchy)
configure_cauchy()
display(widgets.VBox([cauchy_family, cauchy_n, cauchy_factor, cauchy_r, cauchy_button, cauchy_output]))

## Cell 5 — Dini's theorem and the role of compactness and continuity

Dini's theorem upgrades monotone pointwise convergence to uniform convergence when

$$
f_n\in C(K),\qquad K\text{ compact},\qquad
f_n(x)\text{ monotone in }n,qquad f\in C(K).
$$

The weighted powers satisfy every hypothesis. The sequence $x^n$ on $[0,1]$ has a discontinuous pointwise limit. On the non-compact domain $[0,\infty)$, $e^{-x/n}\uparrow1$ pointwise but the global supremum error remains $1$.

In [ ]:
dini_family = widgets.Dropdown(
    options=[
        ("Dini applies: (1-x^2)|x|^n on [-1,1]", "weighted"),
        ("Limit discontinuous: x^n on [0,1]", "powers"),
        ("Domain non-compact: exp(-x/n) on [0,infinity)", "noncompact"),
    ],
    value="weighted",
    description="Example:",
    layout=widgets.Layout(width="520px"),
)
dini_n = widgets.IntSlider(
    value=20, min=1, max=150, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
dini_A = widgets.FloatSlider(
    value=15.0, min=2.0, max=100.0, step=1.0, description="Display A:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
dini_button = style_button(widgets.Button(description="Check hypotheses"), "tasks")
dini_output = widgets.Output()


def run_dini_lab(_=None):
    with dini_output:
        clear_output(wait=True)
        n = dini_n.value
        family = dini_family.value

        if family == "weighted":
            x = np.linspace(-1.0, 1.0, 4001)
            y = (1.0 - x**2) * np.abs(x) ** n
            f = np.zeros_like(x)
            exact = 2.0 / (n + 2.0) * (n / (n + 2.0)) ** (n / 2.0)
            hypotheses = ["compact domain: yes", "continuous f_n and f: yes", "monotone in n: yes", "Dini conclusion: uniform"]
        elif family == "powers":
            x = np.linspace(0.0, 1.0, 4001)
            y = x**n
            f = np.zeros_like(x)
            f[-1] = 1.0
            exact = 1.0
            hypotheses = ["compact domain: yes", "continuous f_n: yes", "monotone in n: yes", "continuous limit: no"]
        else:
            x = np.linspace(0.0, dini_A.value, 4001)
            y = np.exp(-x / n)
            f = np.ones_like(x)
            exact = 1.0
            hypotheses = ["continuous f_n and f: yes", "monotone in n: yes", "compact global domain: no", "global supremum error: 1"]

        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        ax.plot(x, y, color=COLORS["blue"], lw=2, label=rf"$f_{{{n}}}$")
        ax.plot(x, f, color=COLORS["red"], ls="--", label="$f$")
        ax.set_title("Dini's theorem is a package of hypotheses")
        ax.legend()
        plt.show()

        for item in hypotheses:
            print("•", item)
        if family == "weighted":
            print(f"Exact uniform error = {exact:.10g}")
            show_note(r"Dini proves uniform convergence without locating the maximum; exact optimization also gives the rate $\sim2/(en)$.")
        elif family == "powers":
            print(f"Supremum error = {exact:.10g}")
            show_note("The discontinuous-limit test proves non-uniformity immediately.", "#fef2f2", COLORS["red"])
        else:
            displayed_error = 1.0 - math.exp(-dini_A.value / n)
            print(f"Error on displayed compact interval = {displayed_error:.10g}")
            print("Global supremum error                = 1")
            show_note("Compactness is essential. Every fixed compact interval behaves well, but the worst point escapes to infinity.", "#fff7ed", COLORS["orange"])


def configure_dini(change=None):
    dini_A.disabled = dini_family.value != "noncompact"


dini_family.observe(configure_dini, names="value")
dini_button.on_click(run_dini_lab)
configure_dini()
display(widgets.VBox([dini_family, dini_n, dini_A, dini_button, dini_output]))

## Cell 6 — Equicontinuity as protection against narrow peaks

A family is equicontinuous if the same $\delta$ works for every function:

$$
|x-y|<\delta\quad\Longrightarrow\quad
|f_n(x)-f_n(y)|<\varepsilon
\qquad\text{for all }n.
$$

The common modulus

$$
\omega_n(\delta)=\sup_{|x-y|\le\delta}|f_n(x)-f_n(y)|
$$

shows the difference between shrinking oscillations, endpoint layers, and triangular spikes. Equicontinuity plus pointwise convergence on a compact interval upgrades convergence to uniform convergence.

In [ ]:
equi_family = widgets.Dropdown(
    options=[
        ("Equicontinuous: sin(nx)/n", "sine"),
        ("Not equicontinuous: x^n", "powers"),
        ("Not equicontinuous: triangular spikes", "triangle"),
    ],
    value="sine",
    description="Family:",
    layout=widgets.Layout(width="490px"),
)
equi_delta = widgets.FloatLogSlider(
    value=0.02, base=10, min=-4, max=-0.2, step=0.05, description="$\\delta$:",
    continuous_update=False, readout_format=".4f", layout=widgets.Layout(width="490px")
)
equi_N = widgets.IntSlider(
    value=100, min=5, max=500, step=5, description="n up to:",
    continuous_update=False, layout=widgets.Layout(width="490px")
)
equi_button = style_button(widgets.Button(description="Compute modulus"), "arrows-alt-h")
equi_output = widgets.Output()


def modulus_bound(family, n, delta):
    n = np.asarray(n, dtype=float)
    if family == "sine":
        return np.minimum(delta, 2.0 / n)
    if family == "powers":
        return 1.0 - (1.0 - delta) ** n
    return np.minimum(n, 2.0 * n**2 * delta)


def run_equicontinuity(_=None):
    with equi_output:
        clear_output(wait=True)
        family, delta, N = equi_family.value, equi_delta.value, equi_N.value
        n = np.arange(1, N + 1, dtype=float)
        omega = modulus_bound(family, n, delta)

        fig, ax = plt.subplots(figsize=(10.2, 4.1))
        ax.plot(n, omega, color=COLORS["purple"], lw=2)
        ax.set_xlabel("$n$")
        ax.set_ylabel(r"modulus bound at $\delta$")
        ax.set_title(rf"Common-control test at $\delta={delta:.4g}$")
        plt.show()

        print(f"Largest displayed modulus/bound = {np.max(omega):.10g}")
        print(f"Value at n={N}                  = {omega[-1]:.10g}")
        if family == "sine":
            show_note(r"Since $|f_n'(x)|\le1$ for every $n$, one may choose $\delta=\varepsilon$ uniformly. The family is equicontinuous.")
        elif family == "powers":
            show_note(r"For every fixed $\delta>0$, $1-(1-\delta)^n\to1$. No common small modulus exists near $x=1$.", "#fef2f2", COLORS["red"])
        else:
            show_note("The slopes and heights grow with $n$. Narrowing support does not create equicontinuity.", "#fef2f2", COLORS["red"])


equi_button.on_click(run_equicontinuity)
display(widgets.VBox([equi_family, equi_delta, equi_N, equi_button, equi_output]))

## Cell 7 — Locally uniform convergence and escaping error

On a non-compact or non-closed domain, locally uniform convergence requires

$$
\sup_{x\in K}|f_n(x)-f(x)|\longrightarrow0
$$

for every compact $K$ contained in the domain. For $x^n$ on $[0,1)$, every compact set lies in some $[0,r]$ with $r<1$, but the global supremum remains $1$. An escaping Gaussian peak gives the same phenomenon on $\mathbb R$.

In [ ]:
local_family = widgets.Dropdown(
    options=[
        ("Boundary escape: x^n on [0,1)", "powers"),
        ("Spatial escape: exp(-(x-n)^2) on R", "gaussian"),
    ],
    value="powers",
    description="Example:",
    layout=widgets.Layout(width="470px"),
)
local_n = widgets.IntSlider(
    value=20, min=1, max=120, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
local_compact = widgets.FloatSlider(
    value=0.90, min=0.20, max=0.99, step=0.01, description="r:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
local_button = style_button(widgets.Button(description="Compare local/global"), "search-location")
local_output = widgets.Output()


def run_local_uniform(_=None):
    with local_output:
        clear_output(wait=True)
        n = local_n.value
        if local_family.value == "powers":
            r = local_compact.value
            x = np.linspace(0.0, 0.999, 5000)
            values = x**n
            compact_error = r**n
            global_error = 1.0
            compact_label = rf"$[0,{r:.2f}]$"
            ax_limit = None
        else:
            A = local_compact.value
            x = np.linspace(-A, max(A, n + 3.0), 6000)
            values = np.exp(-(x - n) ** 2)
            if n <= A:
                compact_error = 1.0
            else:
                compact_error = math.exp(-(n - A) ** 2)
            global_error = 1.0
            compact_label = rf"$[-{A:.1f},{A:.1f}]$"
            ax_limit = (-A - 1, max(A + 1, n + 2))

        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        ax.plot(x, values, color=COLORS["blue"], lw=2)
        if local_family.value == "powers":
            ax.axvline(local_compact.value, color=COLORS["orange"], ls="--", label="compact boundary")
        else:
            ax.axvspan(-local_compact.value, local_compact.value, color=COLORS["green"], alpha=0.12, label="fixed compact set")
            ax.set_xlim(*ax_limit)
        ax.set_title("The point of largest error escapes the fixed compact set")
        ax.legend()
        plt.show()

        print(f"Supremum error on {compact_label} = {compact_error:.10g}")
        print(f"Global supremum error            = {global_error:.10g}")
        show_note("Local uniform convergence preserves continuity because continuity can be checked on compact neighbourhoods.")


def configure_local(change=None):
    if local_family.value == "powers":
        local_compact.description = "r:"
        local_compact.min, local_compact.max, local_compact.step = 0.20, 0.99, 0.01
        if local_compact.value > 0.99:
            local_compact.value = 0.90
    else:
        local_compact.description = "A:"
        local_compact.min, local_compact.max, local_compact.step = 1.0, 10.0, 0.5
        if local_compact.value < 1.0:
            local_compact.value = 4.0


local_family.observe(configure_local, names="value")
local_button.on_click(run_local_uniform)
configure_local()
display(widgets.VBox([local_family, local_n, local_compact, local_button, local_output]))

## Cell 8 — Passing a limit through an integral

Uniform convergence on $[a,b]$ gives the quantitative estimate

$$
\left|\int_a^b(f_n-f)\right|
\le (b-a)\|f_n-f\|_\infty.
$$

Pointwise convergence alone is insufficient. A moving spike can disappear at every fixed point while retaining nonzero area. Compare a uniformly small sequence with the triangular and smooth boundary spikes from the chapter.

In [ ]:
integral_family = widgets.Dropdown(
    options=[
        ("Uniformly small: x^n/n", "uniform"),
        ("Triangular spike: area 1/2", "triangle"),
        ("Boundary spike: n^2 x(1-x)^n", "boundary"),
    ],
    value="boundary",
    description="Sequence:",
    layout=widgets.Layout(width="500px"),
)
integral_n = widgets.IntSlider(
    value=30, min=1, max=200, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
integral_button = style_button(widgets.Button(description="Integrate sequence"), "area-chart")
integral_output = widgets.Output()


def integration_example(family, n, x):
    if family == "uniform":
        values = x**n / n
        exact_integral = 1.0 / (n * (n + 1.0))
        exact_sup = 1.0 / n
    elif family == "triangle":
        values = triangular_spike(x, n)
        exact_integral = 0.5
        exact_sup = float(n)
    else:
        values = n**2 * x * (1.0 - x) ** n
        exact_integral = n**2 / ((n + 1.0) * (n + 2.0))
        exact_sup = n**2 / (n + 1.0) * (n / (n + 1.0)) ** n
    return values, exact_integral, exact_sup


def run_integration_lab(_=None):
    with integral_output:
        clear_output(wait=True)
        n = integral_n.value
        samples = max(12001, 100 * n)
        x = np.linspace(0.0, 1.0, samples)
        values, exact_integral, exact_sup = integration_example(integral_family.value, n, x)
        cumulative = cumulative_trapezoid(values, x, initial=0.0)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(x, values, color=COLORS["blue"], lw=2)
        if integral_family.value != "uniform":
            axes[0].set_xlim(0, min(1.0, max(6.0 / n, 0.12)))
        axes[0].set_title(r"Integrand $f_n$")
        axes[1].plot(x, cumulative, color=COLORS["green"], lw=2)
        axes[1].axhline(exact_integral, color=COLORS["red"], ls="--", label="exact total integral")
        axes[1].set_title(r"Cumulative integral $F_n(x)$")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Exact supremum norm       = {exact_sup:.10g}")
        print(f"Exact integral over [0,1] = {exact_integral:.10g}")
        print(f"Numerical integral         = {cumulative[-1]:.10g}")
        if integral_family.value == "uniform":
            show_note("Both the supremum error and the integral tend to zero, exactly as the uniform-integration theorem predicts.")
        else:
            show_note("The functions converge pointwise to zero, but not uniformly. Their nonzero limiting area prevents interchange of limit and integral.", "#fef2f2", COLORS["red"])


integral_button.on_click(run_integration_lab)
display(widgets.VBox([integral_family, integral_n, integral_button, integral_output]))

## Cell 9 — Differentiation requires stronger hypotheses

Uniform convergence of $f_n$ does **not** control $f_n'$. A valid differentiation theorem assumes

$$
f_n'\rightrightarrows g
\quad\text{and}\quad
f_n(x_0)\text{ converges at one anchor point}.
$$

It then follows that $f_n\rightrightarrows f$ and $f'=g$. The examples below separate uniform convergence of functions, pointwise or uniform behaviour of derivatives, and the indispensable anchor condition.

In [ ]:
derivative_family = widgets.Dropdown(
    options=[
        ("Valid theorem: x^2 + sin(x)/n", "valid"),
        ("Derivatives oscillate: sin(nx)/n", "oscillatory"),
        ("Derivative boundary layer: exp(-n^2 x^2)/n", "gaussian"),
        ("Missing anchor: f_n(x)=x+n", "anchor"),
    ],
    value="gaussian",
    description="Example:",
    layout=widgets.Layout(width="520px"),
)
derivative_n = widgets.IntSlider(
    value=15, min=1, max=100, step=1, description="n:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
derivative_button = style_button(widgets.Button(description="Compare derivatives"), "superscript")
derivative_output = widgets.Output()


def derivative_example(family, n, x):
    if family == "valid":
        f_n = x**2 + np.sin(x) / n
        f = x**2
        df_n = 2.0 * x + np.cos(x) / n
        df = 2.0 * x
        message = "Both function and derivative errors are at most $1/n$; the theorem applies."
    elif family == "oscillatory":
        f_n = np.sin(n * x) / n
        f = np.zeros_like(x)
        df_n = np.cos(n * x)
        df = np.zeros_like(x)
        message = "The functions converge uniformly to zero, while the derivatives do not converge pointwise in general."
    elif family == "gaussian":
        f_n = np.exp(-(n * x) ** 2) / n
        f = np.zeros_like(x)
        df_n = -2.0 * n * x * np.exp(-(n * x) ** 2)
        df = np.zeros_like(x)
        message = "The derivatives converge pointwise to zero but their supremum remains $\\sqrt{2/e}$."
    else:
        f_n = x + n
        f = x
        df_n = np.ones_like(x)
        df = np.ones_like(x)
        message = "The derivatives agree exactly, but $f_n(0)=n$ has no limit; the anchor condition fails."
    return f_n, f, df_n, df, message


def run_derivative_lab(_=None):
    with derivative_output:
        clear_output(wait=True)
        n = derivative_n.value
        x = np.linspace(-1.0, 1.0, max(5001, 100 * n))
        f_n, f, df_n, df, message = derivative_example(derivative_family.value, n, x)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(x, f_n, color=COLORS["blue"], label="$f_n$")
        axes[0].plot(x, f, color=COLORS["red"], ls="--", label="$f$")
        axes[0].set_title("Functions")
        axes[0].legend()
        axes[1].plot(x, df_n, color=COLORS["purple"], label="$f_n'$")
        axes[1].plot(x, df, color=COLORS["orange"], ls="--", label="candidate limit")
        axes[1].set_title("Derivatives")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Grid sup error for functions   = {np.max(np.abs(f_n-f)):.10g}")
        print(f"Grid sup error for derivatives = {np.max(np.abs(df_n-df)):.10g}")
        if derivative_family.value == "gaussian":
            print(f"Exact derivative supremum      = {math.sqrt(2.0 / math.e):.10g}")
        show_note(message, "#fff7ed" if derivative_family.value != "valid" else "#eff6ff", COLORS["orange"] if derivative_family.value != "valid" else COLORS["blue"])


derivative_button.on_click(run_derivative_lab)
display(widgets.VBox([derivative_family, derivative_n, derivative_button, derivative_output]))

## Cell 10 — Function series and the Weierstrass $M$-test

If $|u_k(x)|\le M_k$ on $E$ and $\sum M_k<\infty$, then

$$
\sum_{k=1}^{\infty}u_k(x)
$$

converges uniformly and absolutely, with the uniform remainder estimate

$$
\left|S(x)-S_N(x)\right|\le\sum_{k=N+1}^{\infty}M_k.
$$

The $M$-test is sufficient, not necessary. The shifted alternating series below converges uniformly on $[0,\infty)$ although it is not absolutely convergent at $x=0$.

In [ ]:
mtest_family = widgets.Dropdown(
    options=[
        ("Geometric function series: sum x^k", "geometric"),
        ("Trigonometric M-test: sum sin(kx)/k^2", "trigonometric"),
        ("Uniform but not absolute: sum (-1)^(k-1)/(k+x)", "alternating"),
    ],
    value="geometric",
    description="Series:",
    layout=widgets.Layout(width="550px"),
)
mtest_N = widgets.IntSlider(
    value=20, min=1, max=300, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="550px")
)
mtest_domain = widgets.FloatSlider(
    value=0.80, min=0.20, max=0.98, step=0.01, description="r/A:",
    continuous_update=False, layout=widgets.Layout(width="550px")
)
mtest_button = style_button(widgets.Button(description="Estimate remainder"), "layer-group")
mtest_output = widgets.Output()


def run_mtest_lab(_=None):
    with mtest_output:
        clear_output(wait=True)
        family, N, parameter = mtest_family.value, mtest_N.value, mtest_domain.value

        if family == "geometric":
            r = parameter
            x = np.linspace(-r, r, 1201)
            k = np.arange(N + 1)
            partial = np.sum(x[:, None] ** k[None, :], axis=1)
            exact = 1.0 / (1.0 - x)
            remainder = np.abs(exact - partial)
            bound = r ** (N + 1) / (1.0 - r)
            comparison = exact
            note = r"The geometric majorant gives a rigorous uniform tail bound. As $r\uparrow1$, the bound deteriorates."
        elif family == "trigonometric":
            x = np.linspace(0.0, 2.0 * math.pi, 1201)
            k = np.arange(1, N + 1)
            partial = np.sum(np.sin(x[:, None] * k[None, :]) / k[None, :] ** 2, axis=1)
            k2 = np.arange(N + 1, 2 * N + 1)
            comparison = partial + np.sum(np.sin(x[:, None] * k2[None, :]) / k2[None, :] ** 2, axis=1)
            remainder = np.abs(comparison - partial)
            bound = 1.0 / N
            note = r"Since $|\sin(kx)|/k^2\le1/k^2$ and the numerical tail is at most $1/N$, the $M$-test proves uniform convergence on all of $\mathbb R$."
        else:
            A = parameter
            x = np.linspace(0.0, A, 1201)
            k = np.arange(1, N + 1)
            partial = np.sum((-1.0) ** (k - 1)[None, :] / (k[None, :] + x[:, None]), axis=1)
            next_term = (-1.0) ** N / (N + 1.0 + x)
            comparison = partial + next_term
            remainder = np.abs(next_term)
            bound = 1.0 / (N + 1.0)
            note = "The alternating remainder is uniform in $x$. At $x=0$, the absolute-value series is harmonic, so the $M$-test cannot be necessary."

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(x, partial, color=COLORS["blue"], lw=2, label="$S_N$")
        axes[0].plot(x, comparison, color=COLORS["orange"], ls="--", label="exact/reference or next approximation")
        axes[0].set_title("Function-series partial sum")
        axes[0].legend()
        axes[1].plot(x, remainder, color=COLORS["purple"], lw=2)
        axes[1].axhline(bound, color=COLORS["red"], ls="--", label="uniform theorem bound")
        axes[1].set_title("Observed remainder quantity")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Uniform theorem bound       = {bound:.10g}")
        print(f"Largest displayed remainder = {np.max(remainder):.10g}")
        show_note(note)


def configure_mtest(change=None):
    if mtest_family.value == "geometric":
        mtest_domain.description = "r:"
        mtest_domain.min, mtest_domain.max, mtest_domain.step = 0.20, 0.98, 0.01
        if mtest_domain.value > 0.98:
            mtest_domain.value = 0.80
    elif mtest_family.value == "alternating":
        mtest_domain.description = "A:"
        use_default_A = mtest_domain.value < 1.0
        mtest_domain.max = 20.0
        mtest_domain.min = 1.0
        mtest_domain.step = 0.5
        if use_default_A:
            mtest_domain.value = 8.0
    else:
        mtest_domain.disabled = True
        return
    mtest_domain.disabled = False


mtest_family.observe(configure_mtest, names="value")
mtest_button.on_click(run_mtest_lab)
configure_mtest()
display(widgets.VBox([mtest_family, mtest_N, mtest_domain, mtest_button, mtest_output]))

## Cell 11 — Uniform Dirichlet convergence through cancellation

On $[\delta,2\pi-\delta]$, the partial sums of $\sin(kx)$ are uniformly bounded:

$$
\left|\sum_{k=1}^{N}\sin(kx)\right|
\le\frac{1}{\sin(\delta/2)}.
$$

Since $1/k\downarrow0$, the uniform Dirichlet test proves uniform convergence of

$$
\sum_{k=1}^{\infty}\frac{\sin(kx)}{k}
$$

on every such closed subinterval. The $M$-test fails because its natural majorant is the divergent harmonic series.

In [ ]:
dirichlet_delta = widgets.FloatSlider(
    value=0.40, min=0.05, max=1.40, step=0.05, description="$\\delta$:",
    continuous_update=False, layout=widgets.Layout(width="490px")
)
dirichlet_N = widgets.IntSlider(
    value=40, min=2, max=500, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="490px")
)
dirichlet_button = style_button(widgets.Button(description="Show cancellation"), "wave-square")
dirichlet_output = widgets.Output()


def sine_series_partial(x, N):
    k = np.arange(1, N + 1, dtype=float)
    return np.sum(np.sin(x[:, None] * k[None, :]) / k[None, :], axis=1)


def run_uniform_dirichlet(_=None):
    with dirichlet_output:
        clear_output(wait=True)
        delta, N = dirichlet_delta.value, dirichlet_N.value
        x = np.linspace(delta, 2.0 * math.pi - delta, 1601)
        S_N = sine_series_partial(x, N)
        S_2N = sine_series_partial(x, 2 * N)
        M = 1.0 / math.sin(delta / 2.0)
        tail_bound = 2.0 * M / (N + 1.0)
        observed = float(np.max(np.abs(S_2N - S_N)))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
        axes[0].plot(x, S_N, color=COLORS["blue"], label="$S_N$")
        axes[0].plot(x, S_2N, color=COLORS["orange"], ls="--", label="$S_{2N}$")
        axes[0].set_title("Partial sums and cancellation")
        axes[0].legend()
        axes[1].plot(x, np.abs(S_2N - S_N), color=COLORS["purple"], lw=2)
        axes[1].axhline(tail_bound, color=COLORS["red"], ls="--", label="Dirichlet tail bound")
        axes[1].set_title("One uniformly controlled tail")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Uniform partial-sum bound M = {M:.10g}")
        print(f"Dirichlet tail bound         = {tail_bound:.10g}")
        print(f"Observed |S_2N-S_N| sup      = {observed:.10g}")
        show_note(r"As $\delta\downarrow0$, the bound blows up. This explains why the theorem is uniform away from the endpoints, not on the whole open interval.")


dirichlet_button.on_click(run_uniform_dirichlet)
display(widgets.VBox([dirichlet_delta, dirichlet_N, dirichlet_button, dirichlet_output]))

## Cell 12 — Varying Riemann--Stieltjes integrators and total variation

For

$$
G_n(x)=\int_a^x f(t)\,d\alpha_n(t),
$$

uniform convergence $\alpha_n\rightrightarrows\alpha$ is generally paired with a common bound on $V_a^b(\alpha_n)$. The controlled example uses

$$
\alpha_n(t)=t+\frac{\sin(nt)}{n^2}.
$$

The lacunary example reproduces the chapter's obstruction: $\|\alpha_m\|_\infty\to0$, but $V(\alpha_m)\to\infty$ and the final integrals remain equal to $\pi$ when the matching frequency is present in $f$.

In [ ]:
rs_family = widgets.Dropdown(
    options=[
        ("Controlled variation: t + sin(nt)/n^2", "controlled"),
        ("Unbounded variation: lacunary counterexample", "lacunary"),
    ],
    value="controlled",
    description="Integrator:",
    layout=widgets.Layout(width="520px"),
)
rs_n = widgets.IntSlider(
    value=8, min=1, max=12, step=1, description="n or m:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
rs_J = widgets.IntSlider(
    value=10, min=2, max=12, step=1, description="Spectral J:",
    continuous_update=False, layout=widgets.Layout(width="520px")
)
rs_button = style_button(widgets.Button(description="Compute RS integrals"), "integral")
rs_output = widgets.Output()


def controlled_rs_data(n, x):
    f = 1.0 + 0.5 * np.cos(x)
    alpha = x
    alpha_n = x + np.sin(n * x) / n**2
    alpha_prime = np.ones_like(x)
    alpha_n_prime = 1.0 + np.cos(n * x) / n
    G = cumulative_trapezoid(f * alpha_prime, x, initial=0.0)
    G_n = cumulative_trapezoid(f * alpha_n_prime, x, initial=0.0)
    variation_bound = 2.0 * math.pi * (1.0 + 1.0 / n)
    return f, alpha, alpha_n, G, G_n, variation_bound


def lacunary_rs_data(m, J, x):
    frequencies = 2.0 ** np.arange(1, J + 1)
    amplitudes = 2.0 ** (-np.arange(1, J + 1) / 2.0)
    f = np.sum(amplitudes[:, None] * np.cos(frequencies[:, None] * x[None, :]), axis=0)
    frequency = 2.0**m
    alpha_n = 2.0 ** (-m / 2.0) * np.sin(frequency * x)
    alpha_n_prime = 2.0 ** (m / 2.0) * np.cos(frequency * x)
    G_n = cumulative_trapezoid(f * alpha_n_prime, x, initial=0.0)
    variation = 4.0 * 2.0 ** (m / 2.0)
    return f, np.zeros_like(x), alpha_n, np.zeros_like(x), G_n, variation


def run_rs_lab(_=None):
    with rs_output:
        clear_output(wait=True)
        family = rs_family.value
        n = rs_n.value
        J = max(rs_J.value, n) if family == "lacunary" else rs_J.value
        if family == "lacunary" and J != rs_J.value:
            rs_J.value = J
        samples = max(12001, 45 * (2**n) if family == "lacunary" else 12001)
        x = np.linspace(0.0, 2.0 * math.pi, samples)

        if family == "controlled":
            f, alpha, alpha_n, G, G_n, variation = controlled_rs_data(n, x)
            sup_alpha = 1.0 / n**2
            expected = "The variations are uniformly bounded, and the cumulative integrals converge uniformly."
        else:
            f, alpha, alpha_n, G, G_n, variation = lacunary_rs_data(n, J, x)
            sup_alpha = 2.0 ** (-n / 2.0)
            expected = "The integrators are uniformly small, but their variations grow; the final integral stays near $\\pi$."

        fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
        axes[0].plot(x, f, color=COLORS["gray"], lw=1.2)
        axes[0].set_title("Fixed integrand approximation $f$")
        axes[1].plot(x, alpha_n, color=COLORS["orange"], lw=1.4, label="$\\alpha_n$")
        axes[1].plot(x, alpha, color=COLORS["red"], ls="--", label="$\\alpha$")
        axes[1].set_title("Integrators")
        axes[1].legend()
        axes[2].plot(x, G_n, color=COLORS["blue"], lw=1.5, label="$G_n$")
        axes[2].plot(x, G, color=COLORS["red"], ls="--", label="$G$")
        axes[2].set_title("Cumulative RS integrals")
        axes[2].legend()
        plt.tight_layout()
        plt.show()

        print(f"Supremum integrator error       = {sup_alpha:.10g}")
        print(f"Variation or variation bound    = {variation:.10g}")
        print(f"Final cumulative integral G_n(b)= {G_n[-1]:.10g}")
        print(f"Supremum cumulative error       = {np.max(np.abs(G_n-G)):.10g}")
        show_note(expected, "#fff7ed" if family == "lacunary" else "#eff6ff", COLORS["orange"] if family == "lacunary" else COLORS["blue"])


def configure_rs(change=None):
    rs_J.disabled = rs_family.value != "lacunary"
    rs_n.description = "m:" if rs_family.value == "lacunary" else "n:"


rs_family.observe(configure_rs, names="value")
rs_button.on_click(run_rs_lab)
configure_rs()
display(widgets.VBox([rs_family, rs_n, rs_J, rs_button, rs_output]))

## Cell 13 — Interactive theorem selector

The correct method depends on the structure of the problem. A rigorous argument should explicitly connect

$$
\text{verified hypotheses}
\quad\Longrightarrow\quad
\text{named theorem}
\quad\Longrightarrow\quad
\text{conclusion}.
$$

Choose the visible structure and inspect the decisive check and the most common warning.

In [ ]:
theorem_cases = {
    "A computable largest error": (
        "Supremum-norm criterion",
        r"Find or bound $\sup_E|f_n-f|$ by a numerical sequence tending to zero.",
        "A maximum on a finite grid is only a lower bound for the true supremum."
    ),
    "A suspected moving peak": (
        "Witness / sequential criterion",
        r"Choose $x_n$ so that $|f_n(x_n)-f(x_n)|$ stays above one fixed positive number.",
        r"The witness point is allowed—and usually required—to depend on $n$."
    ),
    "Late functions are close; limit unknown": (
        "Uniform Cauchy criterion",
        r"Control $\|f_m-f_n\|_\infty$ simultaneously for all $m,n\ge N$.",
        "Checking one selected pair of indices is not enough."
    ),
    "Continuous, monotone, compact setting": (
        "Dini's theorem",
        "Verify compactness, continuity of every function and of the pointwise limit, and a common direction of monotonicity in n.",
        "Monotonicity is in the index; the functions need not be monotone in x."
    ),
    "Pointwise convergence plus common local control": (
        "Equicontinuity upgrade",
        "On a compact interval, verify one modulus of continuity works for the whole family.",
        "Individual continuity does not prevent narrowing spikes."
    ),
    "Every fixed compact subset behaves well": (
        "Locally uniform convergence",
        r"Show $\sup_K|f_n-f|\to0$ for every compact $K$ in the domain.",
        "The global maximum may escape every fixed compact set."
    ),
    "Interchanging limit and integral": (
        "Uniform integration theorem",
        r"Use $|\int(f_n-f)|\le(b-a)\|f_n-f\|_\infty$.",
        "Pointwise convergence alone permits persistent moving mass."
    ),
    "Interchanging limit and derivative": (
        "Uniform derivative theorem",
        r"Require $f_n'\rightrightarrows g$ and convergence at one anchor point.",
        "Uniform convergence of the functions alone says nothing about their derivatives."
    ),
    "A summable numerical majorant": (
        "Weierstrass M-test",
        r"Find $|u_k(x)|\le M_k$ with $\sum M_k<\infty$.",
        "The test is sufficient, not necessary; cancellation may produce uniform convergence."
    ),
    "Uniformly bounded oscillatory partial sums": (
        "Uniform Dirichlet test",
        r"Bound $\sum_{k\le N}a_k(x)$ uniformly and show $b_k(x)\downarrow0$ uniformly.",
        "Pointwise boundedness of the partial sums is not uniform boundedness."
    ),
    "The Riemann--Stieltjes integrator varies": (
        "Varying-integrator theorem",
        r"Combine $\alpha_n\rightrightarrows\alpha$ with a common bound on $V(\alpha_n)$.",
        "Uniformly small integrators can still oscillate with unbounded total variation."
    ),
}

theorem_selector = widgets.Dropdown(
    options=list(theorem_cases.keys()),
    description="Structure:",
    layout=widgets.Layout(width="600px"),
)
theorem_button = style_button(widgets.Button(description="Suggest theorem"), "compass")
theorem_output = widgets.Output()


def suggest_theorem(_=None):
    with theorem_output:
        clear_output(wait=True)
        theorem, check, warning = theorem_cases[theorem_selector.value]
        display(Markdown(f"### Suggested method: **{theorem}**"))
        display(Markdown(f"**Decisive check.** {check}"))
        display(Markdown(f"**Warning.** {warning}"))


theorem_button.on_click(suggest_theorem)
display(widgets.VBox([theorem_selector, theorem_button, theorem_output]))
suggest_theorem()

## Cell 14 — Self-check quiz with immediate feedback

The logical hierarchy is

$$
\text{uniform convergence}\Longrightarrow
\text{pointwise convergence},
$$

but most preservation theorems require additional hypotheses. Answer one question at a time and justify your choice before pressing **Check answer**.

In [ ]:
quiz_questions = [
    {
        "prompt": r"In pointwise convergence, which quantities may the index $N$ depend on?",
        "options": [r"Only $\varepsilon$", r"Both $\varepsilon$ and $x$", r"Only $x$"],
        "answer": r"Both $\varepsilon$ and $x$",
        "explanation": r"Uniform convergence is stronger because its index depends on $\varepsilon$ but not on $x$."
    },
    {
        "prompt": r"What is $\sup_{0\le x<1}x^n$ for a fixed positive integer $n$?",
        "options": ["0", "1/n", "1"],
        "answer": "1",
        "explanation": r"The supremum is $1$ although it is not attained on $[0,1)$."
    },
    {
        "prompt": r"Which missing hypothesis prevents Dini's theorem from applying to $x^n$ on $[0,1]$?",
        "options": ["Compactness", "Continuity of the pointwise limit", "Monotonicity in n"],
        "answer": "Continuity of the pointwise limit",
        "explanation": r"The limit equals $0$ on $[0,1)$ and $1$ at $1$, so it is discontinuous."
    },
    {
        "prompt": r"If $f_n\rightrightarrows f$ on $[a,b]$, which estimate is valid?",
        "options": [r"$|\int(f_n-f)|\le(b-a)\|f_n-f\|_\infty$", r"$|\int(f_n-f)|\ge\|f_n-f\|_\infty$", "No integral estimate follows"],
        "answer": r"$|\int(f_n-f)|\le(b-a)\|f_n-f\|_\infty$",
        "explanation": "This estimate immediately proves convergence of the integrals."
    },
    {
        "prompt": r"Uniform convergence of $f_n'$ alone determines $f_n$ up to what additional information?",
        "options": ["A common maximum", "Convergence at one anchor point", "Pointwise monotonicity"],
        "answer": "Convergence at one anchor point",
        "explanation": "Derivatives do not determine additive constants."
    },
    {
        "prompt": r"What does the Weierstrass $M$-test guarantee?",
        "options": ["Only pointwise convergence", "Uniform and pointwise absolute convergence", "Uniform convergence but never absolute convergence"],
        "answer": "Uniform and pointwise absolute convergence",
        "explanation": r"A summable numerical majorant controls every uniform tail and every absolute-value series."
    },
    {
        "prompt": r"Why is $\alpha_n\rightrightarrows0$ insufficient by itself for varying Riemann--Stieltjes integrators?",
        "options": ["The interval may be compact", "Total variations may be unbounded", "Continuous integrands are forbidden"],
        "answer": "Total variations may be unbounded",
        "explanation": r"Small amplitude can coexist with increasingly rapid oscillation and large variation."
    },
]

quiz_state = {"index": 0, "score": 0, "answered": False}
quiz_prompt = widgets.HTMLMath()
quiz_choice = widgets.ToggleButtons(layout=widgets.Layout(width="100%"))
quiz_check = style_button(widgets.Button(description="Check answer"), "check")
quiz_next = widgets.Button(description="Next question", icon="arrow-right", disabled=True)
quiz_restart = widgets.Button(description="Restart quiz", icon="redo")
quiz_feedback = widgets.Output()
quiz_score = widgets.HTML()


def render_quiz_question():
    q = quiz_questions[quiz_state["index"]]
    quiz_prompt.value = f"<h4>Question {quiz_state['index'] + 1} of {len(quiz_questions)}</h4><p>{q['prompt']}</p>"
    quiz_choice.options = q["options"]
    quiz_choice.value = None
    quiz_state["answered"] = False
    quiz_check.disabled = False
    quiz_next.disabled = True
    quiz_score.value = f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index']} completed"
    with quiz_feedback:
        clear_output()


def check_quiz_answer(_=None):
    if quiz_state["answered"]:
        return
    with quiz_feedback:
        clear_output(wait=True)
        if quiz_choice.value is None:
            display(Markdown("Please choose an answer first."))
            return
        q = quiz_questions[quiz_state["index"]]
        correct = quiz_choice.value == q["answer"]
        if correct:
            quiz_state["score"] += 1
            display(Markdown("### Correct"))
        else:
            display(Markdown(f"### Not yet — the correct answer is **{q['answer']}**"))
        display(Markdown(q["explanation"]))
    quiz_state["answered"] = True
    quiz_check.disabled = True
    quiz_next.disabled = False
    quiz_score.value = f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index'] + 1} completed"


def next_quiz_question(_=None):
    if quiz_state["index"] < len(quiz_questions) - 1:
        quiz_state["index"] += 1
        render_quiz_question()
    else:
        with quiz_feedback:
            clear_output(wait=True)
            display(Markdown(f"## Quiz complete: {quiz_state['score']} / {len(quiz_questions)}"))
        quiz_next.disabled = True


def restart_quiz(_=None):
    quiz_state.update(index=0, score=0, answered=False)
    render_quiz_question()


quiz_check.on_click(check_quiz_answer)
quiz_next.on_click(next_quiz_question)
quiz_restart.on_click(restart_quiz)
render_quiz_question()
display(widgets.VBox([
    quiz_prompt,
    quiz_choice,
    widgets.HBox([quiz_check, quiz_next, quiz_restart]),
    quiz_score,
    quiz_feedback,
]))

## Cell 15 — Practice generator: theorem, hypotheses, conclusion

For each problem, write a solution in the form

$$
\text{pointwise limit}
\;\longrightarrow\;
\text{supremum estimate or theorem}
\;\longrightarrow\;
\text{classification and consequences}.
$$

Press **New problem**, attempt a proof, and reveal the hint or solution only afterward.

In [ ]:
practice_bank = [
    {
        "problem": r"Study $f_n(x)=\dfrac{nx}{n+x}$ on $[0,A]$ and on $[0,\infty)$.",
        "hint": r"The error is $x^2/(n+x)$ and is increasing in $x$.",
        "solution": r"On $[0,A]$, the exact error is $A^2/(n+A)\to0$, so convergence to $x$ is uniform. On $[0,\infty)$ the error is unbounded, so convergence is not uniform."
    },
    {
        "problem": r"Determine whether $f_n(x)=(1-x^2)|x|^n$ converges uniformly on $[-1,1]$.",
        "hint": "Check the hypotheses of Dini's theorem before maximizing.",
        "solution": r"The functions are continuous, decrease pointwise to the continuous zero function on a compact interval, so Dini gives uniform convergence. The exact error is $\frac{2}{n+2}(\frac{n}{n+2})^{n/2}$."
    },
    {
        "problem": r"Explain why $x^n\to0$ locally uniformly but not uniformly on $[0,1)$.",
        "hint": r"Every compact subset is contained in $[0,r]$ for some $r<1$.",
        "solution": r"On $[0,r]$ the error is at most $r^n\to0$, while $\sup_{0\le x<1}x^n=1$ for every $n$."
    },
    {
        "problem": r"Can one interchange limit and integral for $f_n(x)=n^2x(1-x)^n$ on $[0,1]$?",
        "hint": r"Compute the pointwise limit and the beta-type integral separately.",
        "solution": r"The pointwise limit is zero, but $\int_0^1f_n=n^2/[(n+1)(n+2)]\to1$. Thus the interchange fails; convergence is not uniform."
    },
    {
        "problem": r"Study the derivatives of $f_n(x)=n^{-1}e^{-n^2x^2}$ on $[-1,1]$.",
        "hint": r"Use the moving point $x_n=1/(\sqrt2n)$.",
        "solution": r"The functions converge uniformly to zero. The derivatives converge pointwise to zero, but $|f_n'(x_n)|=\sqrt{2/e}$, so derivative convergence is not uniform."
    },
    {
        "problem": r"Prove uniform convergence of $\sum_{k=1}^{\infty}\sin(kx)/k^2$ on $\mathbb R$.",
        "hint": r"Use a numerical majorant independent of $x$.",
        "solution": r"Since $|\sin(kx)|/k^2\le1/k^2$ and $\sum1/k^2$ converges, the Weierstrass $M$-test gives uniform and absolute convergence."
    },
    {
        "problem": r"Why does $\sum_{k=1}^{\infty}(-1)^{k-1}/(k+x)$ converge uniformly on $[0,\infty)$ although the $M$-test fails?",
        "hint": "Use the alternating remainder uniformly in x.",
        "solution": r"The remainder is at most $1/(N+1+x)\le1/(N+1)$. At $x=0$, the absolute-value series is harmonic, so no summable $M$-test majorant based on the exact suprema exists."
    },
    {
        "problem": r"What additional control is needed when $\alpha_n\rightrightarrows\alpha$ in cumulative Riemann--Stieltjes integrals?",
        "hint": "Rapid oscillation can have small amplitude but large total variation.",
        "solution": r"For a general continuous integrand, require a uniform bound on $V_a^b(\alpha_n)$. Without it, uniformly small integrators can produce non-vanishing integrals."
    },
]

practice_rng = random.Random(2026)
practice_state = {"item": None}
practice_output = widgets.Output()
new_problem_button = style_button(widgets.Button(description="New problem"), "dice")
hint_button = widgets.Button(description="Show hint", icon="lightbulb")
solution_button = widgets.Button(description="Reveal solution", icon="eye")


def new_practice_problem(_=None):
    practice_state["item"] = practice_rng.choice(practice_bank)
    with practice_output:
        clear_output(wait=True)
        display(Markdown("### Problem"))
        display(Markdown(practice_state["item"]["problem"]))
        display(Markdown("_State the pointwise limit and every theorem hypothesis before classifying the convergence._"))


def show_practice_hint(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Hint.** {practice_state['item']['hint']}"))


def show_practice_solution(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Solution.** {practice_state['item']['solution']}"))


new_problem_button.on_click(new_practice_problem)
hint_button.on_click(show_practice_hint)
solution_button.on_click(show_practice_solution)
display(widgets.VBox([
    widgets.HBox([new_problem_button, hint_button, solution_button]),
    practice_output,
]))
new_practice_problem()

## Suggested learning path and extensions

1. In the supremum laboratory, move the domain endpoint for $x^n$ from $r=0.8$ to $r=1$. Explain the qualitative change.
2. Keep the triangular-spike grid fixed and increase $n$. Record when the sampled maximum becomes zero while the true maximum grows.
3. In the uniform Cauchy laboratory, use $m=2n$ for $x^n$ on $[0,1]$ and derive the exact value $1/4$ analytically.
4. Compare Dini's valid example with each failed hypothesis example. State precisely which assumption is missing.
5. For the integration laboratory, compare the limiting pointwise function, supremum norm, and total mass of each family.
6. In the function-series laboratory, explain why a sufficient test may fail even when uniform convergence is true.
7. Let $\delta$ approach zero in the uniform Dirichlet experiment and interpret the growth of $1/\sin(\delta/2)$.
8. In the Riemann--Stieltjes laboratory, compare the decay of $\|\alpha_n-\alpha\|_\infty$ with the growth or boundedness of total variation.

### Final reminder

A finite graph can suggest a candidate limit, but uniform convergence is the global statement

$$
\sup_{x\in E}|f_n(x)-f(x)|\longrightarrow0.
$$

Exact estimates, theorem hypotheses, and witness sequences turn the visual evidence into proof.